[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/08_real_model_lab/08_real_model_lab.ipynb)

# 08 · 真模型实验室：用 Gemini 复跑全课实验（BONUS）

<span style="background:#7c3aed;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">API</span> 检测到环境变量 `GEMINI_API_KEY` / `GOOGLE_API_KEY` 时使用**真实 Gemini API**（纯标准库 `urllib`，不依赖任何 SDK）；没有 key 时自动回退到内置 **MockGemini**（确定性模拟模型，rng 可复现）。两条路径接口完全相同，**整本 notebook 无 key 也能从头跑到尾**。

00–07 的全部实验跑在纯 numpy 模拟推理器上——可控、可重复、有解析解对照（internal validity）。本模块把其中的关键实验在真模型上复跑一遍，检验模拟结论的**外部效度**（external validity）。对照的是**定性形状**（上升 / 饱和 / 掉点），不是数值。

**本 notebook 你将完成：**

1. **统一 API 层**：`llm_generate(prompt, n, temperature, thinking_budget)` —— urllib 调 `generateContent` + 磁盘缓存 + 指数退避重试 + mock 优雅回退；
2. **题库与答案抽取**：20 道模板化数学题（换参数可重算答案）+ 鲁棒 `extract_answer`；
3. **实验 A（对应 02）**：self-consistency 的 majority@N 曲线；
4. **实验 B（对应 03）**：LLM-as-verifier 的 Best-of-N vs majority vs random；
5. **实验 C（对应 06）**：thinkingBudget 三档扫描——"简单题早停无损、难题需要长思考"；
6. **实验 D（对应 07）**：GSM-Symbolic 式扰动（换数字 / 加无关从句）的掉点测量；
7. **汇总**：模拟预测 vs 真模型观测对照表（含 05 的 scaling 视角）+ token 成本核算；
8. **3 道 ✏️ 练习**（assert 自动判分，全部走 mock / 纯函数，不依赖网络）。

**成本预估**：默认参数下全部实验约 $10^5$–$10^6$ token，gemini-2.5-flash 档 **< $1**；跑实验前会先打印粗估成本——先算账再花钱是 API 评测的第一纪律。mock 路径完全免费。

参考：[Wang 2022, arXiv:2203.11171] · [Cobbe 2021, arXiv:2110.14168] · [Snell 2024, arXiv:2408.03314] · [Muennighoff 2025, arXiv:2501.19393] · [Mirzadeh 2024, arXiv:2410.05229] · Gemini API thinking 文档

## 1 · 统一 API 层：真 Gemini 与 MockGemini 同接口

设计要点（对应讲解第 3 节的四条工程纪律）：

- **REST 直调**：`POST .../v1beta/models/{model}:generateContent`，body 里 `generationConfig` 支持 `temperature` / `candidateCount`（一次最多 8 条采样）/ `maxOutputTokens` / `thinkingConfig.thinkingBudget`。key 从环境变量读取，**绝不进代码**，异常信息里也抹掉。
- **磁盘缓存**：key = `sha256(model + prompt + 全部采样参数)`。断点续跑不重复计费；**任何影响输出的参数都必须进 key**，否则会拿错缓存（静默 bug）。
- **重试**：429/5xx 指数退避（2s → 4s），最多 3 次。
- **优雅回退**：`MockGemini` 对注册过的题目按"难度 × 思考预算档位"参数化正确率作答，错误答案带**系统性偏差**（固定 distractor 吃掉大部分错误质量）——它就是 00–07 的模拟推理器穿上 API 的衣服，因此 mock 路径会精确复现模拟版结论。

In [ ]:
import os, json, time, math, hashlib, random, re, urllib.request, urllib.error
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL     = "gemini-2.5-flash"
API_KEY   = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
USE_REAL_API = API_KEY is not None
CACHE_DIR = "_gemini_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
USAGE    = {"calls": 0, "cache_hits": 0, "prompt_tokens": 0, "output_tokens": 0, "thinking_tokens": 0}
REGISTRY = {}   # question_text -> {"answer", "difficulty", "penalty"}，第 2 节填充；mock 靠它"会做题"

def _call_gemini_real(prompt, n, temperature, thinking_budget, max_output_tokens=1024):
    """纯 urllib 调 Gemini REST API，带指数退避重试。返回 (texts, usage)。"""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    gen_cfg = {"temperature": temperature, "candidateCount": min(n, 8),
               "maxOutputTokens": max_output_tokens}
    if thinking_budget is not None:
        gen_cfg["thinkingConfig"] = {"thinkingBudget": int(thinking_budget)}
    body = json.dumps({"contents": [{"parts": [{"text": prompt}]}],
                       "generationConfig": gen_cfg}).encode("utf-8")
    req = urllib.request.Request(url, data=body, method="POST",
        headers={"Content-Type": "application/json", "x-goog-api-key": API_KEY})
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                data = json.loads(resp.read())
            texts = []
            for cand in data.get("candidates", []):
                parts = cand.get("content", {}).get("parts", [])
                texts.append("".join(p.get("text", "") for p in parts))
            um = data.get("usageMetadata", {})
            usage = {"prompt": um.get("promptTokenCount", 0),
                     "output": um.get("candidatesTokenCount", 0),
                     "thinking": um.get("thoughtsTokenCount", 0)}
            return texts, usage
        except urllib.error.HTTPError as e:           # 注意：异常里不带 URL，避免泄漏 key
            if e.code in (429, 500, 503) and attempt < 2:
                time.sleep(2 ** (attempt + 1)); continue
            raise RuntimeError(f"Gemini API HTTP {e.code}（key 已隐去）") from None
    raise RuntimeError("Gemini API 重试 3 次仍失败")

def _budget_level(tb):
    """把 thinkingBudget 粗分成三档（mock 用；None=动态思考，当作充足预算）。"""
    if tb is None: return "high"
    if tb == 0:    return "low"
    return "mid" if tb <= 1024 else "high"

def fmt_num(x):
    x = float(x)
    return str(int(x)) if x == int(x) else f"{x:g}"

class MockGemini:
    """确定性模拟模型：对 REGISTRY 里注册过的题，按 难度×预算档×扰动惩罚 的正确率作答。
    rng 种子 = hash(prompt+参数+样本序号)，跨进程完全可复现。"""
    BASE_P = {"easy": 0.92, "medium": 0.72, "hard": 0.42}
    BUDGET_FACTOR = {"low":  {"easy": 1.00, "medium": 0.75, "hard": 0.45},   # 简单题早停无损
                     "mid":  {"easy": 1.00, "medium": 0.95, "hard": 0.80},
                     "high": {"easy": 1.00, "medium": 1.00, "hard": 1.00}}
    BIAS = {"easy": 0.4, "medium": 0.5, "hard": 0.75}   # 错误质量集中到固定 distractor 的比例
    MOCK_THINK_TOKENS = {"low": 8, "mid": 96, "high": 256}

    def generate(self, prompt, n, temperature, thinking_budget):
        texts = []
        for i in range(n):
            tag = 0 if temperature == 0 else i      # temperature=0 → 贪心，n 条全相同
            seed = int(hashlib.md5(f"{prompt}|{tag}|{temperature}|{thinking_budget}".encode()).hexdigest()[:8], 16)
            texts.append(self._one(prompt, random.Random(seed), temperature, thinking_budget))
        level = _budget_level(thinking_budget)
        usage = {"prompt": len(prompt) // 4,
                 "output": sum(len(t) // 4 for t in texts),
                 "thinking": self.MOCK_THINK_TOKENS[level] * n}
        return texts, usage

    def _match(self, prompt):
        hits = [q for q in REGISTRY if q in prompt]
        return REGISTRY[max(hits, key=len)] if hits else None   # 最长匹配：noop 变体优先于原题

    def _one(self, prompt, rng, temperature, thinking_budget):
        meta = self._match(prompt)
        if "[VERIFIER]" in prompt:
            return self._verify(prompt, meta, rng)
        if meta is None:
            return "（mock：题目未注册，无法作答。）答案是 0。"
        level = _budget_level(thinking_budget)
        d = meta["difficulty"]
        p = self.BASE_P[d] * self.BUDGET_FACTOR[level][d] * meta["penalty"]
        truth = float(meta["answer"])
        correct = (p >= 0.5) if temperature == 0 else (rng.random() < p)
        if correct:
            ans = truth
        elif rng.random() < self.BIAS[d]:
            ans = truth + 1                          # 系统性偏差：固定 distractor（majority 的克星）
        else:
            ans = truth + rng.choice([-3, -2, -1, 2, 3])
        steps = {"low": 1, "mid": 3, "high": 6}[level]
        return f"思考：分 {steps} 步推理（mock 模拟，难度 {d}）。答案是 {fmt_num(ans)}。"

    def _verify(self, prompt, meta, rng):
        """模拟 LLM verifier：答案对 → 高分（8.5±1.2），错 → 低分（3.5±1.8），有噪声、不完美。"""
        if meta is None: return "分数: 1"
        cand = extract_answer(prompt.split("候选解答：")[-1])    # extract_answer 在第 2 节定义
        ok = cand is not None and abs(cand - float(meta["answer"])) < 1e-6
        mu, sd = (8.5, 1.2) if ok else (3.5, 1.8)
        return f"分数: {int(min(10, max(1, round(rng.gauss(mu, sd)))))}"

MOCK = MockGemini()

def _cache_key(prompt, n, temperature, thinking_budget):
    blob = json.dumps({"model": MODEL if USE_REAL_API else "mock", "prompt": prompt, "n": n,
                       "temperature": temperature, "thinking_budget": thinking_budget},
                      sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:24]

def llm_generate(prompt, n=1, temperature=1.0, thinking_budget=None):
    """统一入口：真 Gemini / MockGemini 完全同接口。返回 list[str]（n 条采样），带磁盘缓存。"""
    path = os.path.join(CACHE_DIR, _cache_key(prompt, n, temperature, thinking_budget) + ".json")
    if os.path.exists(path):
        with open(path) as f:
            rec = json.load(f)
        USAGE["cache_hits"] += 1
        return rec["texts"]
    if USE_REAL_API:
        texts, usage = [], {"prompt": 0, "output": 0, "thinking": 0}
        remaining = n
        while remaining > 0:                          # candidateCount 上限 8，分批
            t, u = _call_gemini_real(prompt, min(remaining, 8), temperature, thinking_budget)
            texts += t
            for k in usage: usage[k] += u[k]
            remaining -= min(remaining, 8)
    else:
        texts, usage = MOCK.generate(prompt, n, temperature, thinking_budget)
    USAGE["calls"] += 1
    USAGE["prompt_tokens"]   += usage["prompt"]
    USAGE["output_tokens"]   += usage["output"]
    USAGE["thinking_tokens"] += usage["thinking"]
    with open(path, "w") as f:
        json.dump({"texts": texts, "usage": usage}, f, ensure_ascii=False)
    return texts

print("后端:", f"真实 Gemini API（{MODEL}）" if USE_REAL_API
      else "MockGemini —— 未检测到 GEMINI_API_KEY / GOOGLE_API_KEY，离线确定性模拟")

## 2 · 题库与答案抽取

20 道有唯一数值答案的题：8 道自写的 GSM8K 风格小学应用题 + 12 道数字谜题，难度标签 easy/medium/hard（7/7/6）。每题**模板化**——`template + params + answer_fn`，换参数即可重算答案，这正是实验 D（GSM-Symbolic 式扰动）需要的基础设施 [Mirzadeh 2024]。

`extract_answer` 用两级策略（衔接 LLM_Evals_Course 03 与本课 07 的判分器一节）：先找 `答案是/答案为/answer is X` 模式（取最后一次出现），再退化到**全文最后一个数字**；都失败显式返回 `None` 而不是算错。**先验证抽取器、再跑实验**——抽取误差会污染所有下游结论。

每题的三个版本（原版 / 换数字 / 加无关从句）都注册进 `REGISTRY`，mock 对扰动版施加正确率惩罚（×0.85 / ×0.70），模拟"模板依赖"的脆弱性；真 API 路径下 REGISTRY 只用来判分，惩罚不起作用。

In [ ]:
def P(template, answer_fn, difficulty, params, alt_params):
    return {"template": template, "answer_fn": answer_fn, "difficulty": difficulty,
            "params": params, "alt_params": alt_params,
            "question": template.format(**params), "answer": float(answer_fn(**params))}

PROBLEMS = [
    # ---- GSM8K 风格应用题（8 道）----
    P("小明有 {a} 个苹果，又买了 {b} 个，吃掉 {c} 个，还剩多少个？",
      lambda a, b, c: a + b - c, "easy",   dict(a=7, b=5, c=3),   dict(a=9, b=6, c=4)),
    P("一支铅笔 {a} 元，小红买了 {b} 支，一共花多少元？",
      lambda a, b: a * b,        "easy",   dict(a=3, b=6),        dict(a=4, b=7)),
    P("停车场原有 {a} 辆车，开走 {b} 辆，又开来 {c} 辆，现在有多少辆？",
      lambda a, b, c: a - b + c, "easy",   dict(a=15, b=6, c=8),  dict(a=21, b=9, c=5)),
    P("班里有 {a} 名学生，每 {b} 人一组，最多能分成多少个完整的组？",
      lambda a, b: a // b,       "medium", dict(a=37, b=5),       dict(a=44, b=7)),
    P("一辆汽车每小时行驶 {a} 千米，行驶 {b} 小时后距终点还有 {c} 千米，全程多少千米？",
      lambda a, b, c: a * b + c, "medium", dict(a=60, b=3, c=40), dict(a=55, b=4, c=30)),
    P("水池里有 {a} 升水，每分钟同时流出 {b} 升、流入 {c} 升，多少分钟后正好有 {d} 升？",
      lambda a, b, c, d: (d - a) / (c - b), "medium", dict(a=20, b=2, c=5, d=50), dict(a=30, b=3, c=7, d=58)),
    P("鸡兔同笼，共有 {h} 个头、{f} 只脚，兔子有多少只？",
      lambda h, f: (f - 2 * h) / 2, "hard", dict(h=10, f=28),     dict(h=12, f=40)),
    P("某商品价格先上涨 {a}%，再下降 {a}%，原价 {p} 元，现价多少元？",
      lambda a, p: p * (1 - (a / 100) ** 2), "hard", dict(a=20, p=100), dict(a=10, p=200)),
    # ---- 数字谜题（12 道）----
    P("计算：{a} + {b} × {c} = ?",          lambda a, b, c: a + b * c, "easy",   dict(a=7, b=6, c=5),   dict(a=8, b=9, c=4)),
    P("从 {a} 数到 {b}（两端都算），一共数了多少个数？", lambda a, b: b - a + 1, "easy", dict(a=13, b=57), dict(a=24, b=91)),
    P("{a} 的一半加上 {b} 等于多少？",        lambda a, b: a / 2 + b,    "easy",   dict(a=26, b=9),       dict(a=48, b=7)),
    P("计算：{a} - {b} + {c} = ?",          lambda a, b, c: a - b + c, "easy",   dict(a=52, b=17, c=8), dict(a=61, b=29, c=5)),
    P("一个等差数列首项是 {a}，公差是 {d}，第 {n} 项是多少？",
      lambda a, d, n: a + (n - 1) * d, "medium", dict(a=3, d=4, n=10),  dict(a=5, d=3, n=12)),
    P("三个连续整数的和是 {s}，其中最大的数是多少？", lambda s: s // 3 + 1, "medium", dict(s=45), dict(s=72)),
    P("一个两位数，十位数字是个位数字的 {k} 倍，两位数字之和是 {s}，这个两位数是多少？",
      lambda k, s: 10 * k * (s // (k + 1)) + s // (k + 1), "hard", dict(k=2, s=9), dict(k=3, s=8)),
    P("{a} 除以 {b}，余数是多少？",          lambda a, b: a % b,        "medium", dict(a=38, b=7),       dict(a=52, b=9)),
    P("1 到 {n} 的所有整数之和是多少？",      lambda n: n * (n + 1) // 2, "medium", dict(n=40),           dict(n=60)),
    P("某数的 {a} 倍减去 {b} 等于 {c}，这个数是多少？", lambda a, b, c: (c + b) / a, "hard", dict(a=4, b=7, c=25), dict(a=6, b=5, c=43)),
    P("甲乙两数之和是 {s}，差是 {d}（甲大于乙），甲是多少？", lambda s, d: (s + d) / 2, "hard", dict(s=50, d=14), dict(s=76, d=22)),
    P("把 {n} 个糖果分给若干小朋友，每人分 {k} 个则剩 {r} 个，有多少个小朋友？",
      lambda n, k, r: (n - r) / k, "hard", dict(n=38, k=5, r=3),  dict(n=47, k=6, r=5)),
]
DIFFS = ["easy", "medium", "hard"]

# ---- 三个版本全部注册进 REGISTRY（mock 的"题感"+ 实验 D 的判分依据）----
NOOP_CLAUSE = "（补充：当天是星期三，旁边还停着 7 辆自行车，这些信息与本题无关。）"
REGISTRY.clear()
for prob in PROBLEMS:
    prob["q_swap"] = prob["template"].format(**prob["alt_params"])
    prob["a_swap"] = float(prob["answer_fn"](**prob["alt_params"]))
    prob["q_noop"] = prob["question"] + NOOP_CLAUSE
    REGISTRY[prob["question"]] = {"answer": prob["answer"], "difficulty": prob["difficulty"], "penalty": 1.0}
    REGISTRY[prob["q_swap"]]   = {"answer": prob["a_swap"], "difficulty": prob["difficulty"], "penalty": 0.85}
    REGISTRY[prob["q_noop"]]   = {"answer": prob["answer"], "difficulty": prob["difficulty"], "penalty": 0.70}

def solve_prompt(question):
    return f"解下面的数学题：先简要推理，最后一行必须用'答案是 X'的格式给出数值答案。\n题目：{question}"

def extract_answer(text):
    """鲁棒数值抽取：先找'答案是/答案为/answer is X'（取最后一次），再退化到最后一个数字；失败返回 None。"""
    if not text:
        return None
    t = text.replace(",", "")                       # 千分位逗号
    m = re.findall(r"(?:答案是|答案为|answer is)\s*[:：]?\s*(-?\d+(?:\.\d+)?)", t, flags=re.IGNORECASE)
    if m:
        return float(m[-1])
    nums = re.findall(r"-?\d+(?:\.\d+)?", t)
    return float(nums[-1]) if nums else None

def is_correct(ans, truth):
    return ans is not None and math.isclose(ans, float(truth), rel_tol=1e-6, abs_tol=1e-6)

# ---- 先验证抽取器，再跑实验 ----
assert extract_answer("经过计算，答案是 42。") == 42.0
assert extract_answer("答案为 -3.5，搞定") == -3.5
assert extract_answer("结果等于 17") == 17.0           # 退化路径
assert extract_answer("这题没有数字。") is None
assert len(PROBLEMS) == 20 and [sum(p["difficulty"] == d for p in PROBLEMS) for d in DIFFS] == [7, 7, 6]

# ---- 成本预估先行（第一纪律）：跑之前先算账 ----
PRICE_PER_1K = {"in": 0.0003, "out": 0.0025}   # USD/1k tokens，gemini-2.5-flash（2025 价目，请核对官网）
def estimate_tokens(text): return max(1, len(text) // 4)
def cost_estimate(prompts, n_samples=1, est_output_tokens=300,
                  price_in=PRICE_PER_1K["in"], price_out=PRICE_PER_1K["out"]):
    t_in  = sum(estimate_tokens(p) for p in prompts) * n_samples
    t_out = est_output_tokens * len(prompts) * n_samples
    return t_in / 1000 * price_in + t_out / 1000 * price_out

total_samples_per_q = 16 + 8 + 3 * 6 + 3 * 4    # 实验 A + B(verifier) + C + D 的量级
est = cost_estimate([solve_prompt(p["question"]) for p in PROBLEMS], n_samples=total_samples_per_q)
print(f"全部实验粗估成本 ≈ ${est:.3f}（flash 档；mock 路径 $0）")

demo = llm_generate(solve_prompt(PROBLEMS[0]["question"]), n=2, temperature=1.0)
print("示例输出:", demo[0])
print("抽取:", extract_answer(demo[0]), "| 真值:", PROBLEMS[0]["answer"])

## 3 · 实验 A（对应 02）：self-consistency 的 majority@N 曲线

**模拟预测**（02 模块）：majority@N 随 N 上升并收敛 [Wang 2022]；但若模型对某题有**系统性偏差**（最频繁的错误答案出现率超过正确答案），majority 会收敛到错误答案——这种题加算力救不了。

**设计**：每题 `temperature=1.0` 采 16 条（多样性来自温度采样——temperature=0 采 16 条全一样，曲线必然是平线），取前 N 条做 majority voting，画 accuracy vs N 并按难度分层。**看什么**：整体是否"上升收敛"；hard 层是否出现 plateau（被偏差卡住的现实版）。

In [ ]:
# 实验 A：majority@N（每题 16 条采样会被磁盘缓存，重跑零成本）
N_MAX = 16
samples = {p["question"]: llm_generate(solve_prompt(p["question"]), n=N_MAX, temperature=1.0)
           for p in PROBLEMS}
all_ans = {q: [extract_answer(t) for t in ts] for q, ts in samples.items()}

def majority_vote(answers):
    vals = [a for a in answers if a is not None]
    return Counter(vals).most_common(1)[0][0] if vals else None

Ns = [1, 2, 4, 8, 16]
rows = []
for n in Ns:
    for prob in PROBLEMS:
        maj = majority_vote(all_ans[prob["question"]][:n])
        rows.append({"N": n, "difficulty": prob["difficulty"],
                     "correct": is_correct(maj, prob["answer"])})
df_a = pd.DataFrame(rows)
overall_a = df_a.groupby("N")["correct"].mean()
strat_a = df_a.groupby(["N", "difficulty"])["correct"].mean().unstack()[DIFFS]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(overall_a.index, overall_a.values, "k-o", lw=2.5, label="overall")
for d, mk in zip(DIFFS, ["s", "^", "v"]):
    ax.plot(strat_a.index, strat_a[d], "--", marker=mk, alpha=.75, label=d)
ax.set_xscale("log", base=2); ax.set_xticks(Ns); ax.set_xticklabels(Ns)
ax.set_xlabel("N (samples)"); ax.set_ylabel("majority@N accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("Experiment A: self-consistency, real-model version (cf. Module 02)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
print(strat_a.round(2))
print("定性对照：easy/medium 上升收敛 ✓；hard 若 plateau → 系统性偏差的现实版")

## 4 · 实验 B（对应 03）：LLM-as-verifier 的 Best-of-N

**模拟预测**（03 模块）：verifier 好于随机时 BoN > random；BoN 与 majority 的相对强弱取决于 verifier 质量与错误答案的分布。这正是 GSM8K verifier 的原始设计 [Cobbe 2021]——只是当年训了个专用 ORM，这里直接用低 temperature 的 Gemini 当 zero-shot verifier。

**设计**：复用实验 A 的前 8 条采样作候选；verifier prompt 要求只输出 `分数: s`（1–10）；三线对比 **BoN（取最高分）/ majority@8 / random（取第 1 条）**。**看什么**：真 LLM verifier 是否提供超过 majority 的信号；顺手翻一翻 verifier 给错误解答打高分的案例——那是 Goodhart 过优化（03 模块）的现实素材。

In [ ]:
# 实验 B：BoN(verifier) vs majority vs random
def verifier_prompt(question, solution):
    return ("[VERIFIER] 你是严格的数学阅卷员。判断下面候选解答的最终答案是否正确，"
            "只输出一行，格式为'分数: s'，s 是 1-10 的整数（10 = 确定正确）。\n"
            f"题目：{question}\n候选解答：{solution}")

K = 8
rows = []
for prob in PROBLEMS:
    cands   = samples[prob["question"]][:K]
    answers = all_ans[prob["question"]][:K]
    scores = []
    for text in cands:
        out = llm_generate(verifier_prompt(prob["question"], text), n=1, temperature=0.0)[0]
        m = re.search(r"分数\s*[:：]\s*(\d+)", out)
        scores.append(int(m.group(1)) if m else 0)   # 解析失败记 0 分（显式而非崩溃）
    rows.append({"difficulty": prob["difficulty"],
                 "BoN(verifier)": is_correct(answers[int(np.argmax(scores))], prob["answer"]),
                 "majority@8":    is_correct(majority_vote(answers), prob["answer"]),
                 "random(1)":     is_correct(answers[0], prob["answer"])})
df_b = pd.DataFrame(rows)
print("总体准确率：")
print(df_b[["BoN(verifier)", "majority@8", "random(1)"]].mean().round(2).to_string())
print("\n按难度分层：")
print(df_b.groupby("difficulty")[["BoN(verifier)", "majority@8", "random(1)"]].mean().reindex(DIFFS).round(2))
print("\n定性对照：BoN ≥ majority > random ⇒ verifier 有真信号（03 模块的现实版）")

## 5 · 实验 C（对应 06）：thinking budget 扫描

**模拟预测**（06 模块）：简单题在很小的思考预算下就饱和（**早停无损**），难题准确率随预算上升（**underthinking 有代价**）——这条不对称性是按题分配预算的 compute-optimal 路由 [Snell 2024] 的依据。

**设计**：`thinkingConfig.thinkingBudget` ∈ {0, 512, 2048}（0 = 关闭思考，flash 档支持）——这是 s1 budget forcing [Muennighoff 2025] 的 API 原语化。每档每题采 4 条，按难度分层画 accuracy vs budget。模型不支持 thinkingConfig 时的退化方案：`maxOutputTokens` 截断 + 用"直接给答案"/"展示完整推理"的 prompt 控制思考长度。**注意**：真 API 的 budget 是软目标，正式报告要记实际 `thoughtsTokenCount` 而非名义预算。

In [ ]:
# 实验 C：accuracy vs thinkingBudget，按难度分层
BUDGETS = [0, 512, 2048]
acc_c = {d: [] for d in DIFFS}
for tb in BUDGETS:
    per = {d: [] for d in DIFFS}
    for prob in PROBLEMS:
        texts = llm_generate(solve_prompt(prob["question"]), n=6, temperature=1.0, thinking_budget=tb)
        per[prob["difficulty"]].append(np.mean([is_correct(extract_answer(t), prob["answer"]) for t in texts]))
    for d in DIFFS:
        acc_c[d].append(float(np.mean(per[d])))

fig, ax = plt.subplots(figsize=(7, 4))
for d, mk in zip(DIFFS, ["o", "s", "^"]):
    ax.plot(BUDGETS, acc_c[d], marker=mk, lw=2, label=d)
ax.set_xlabel("thinkingBudget (tokens)"); ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("Experiment C: accuracy vs thinking budget (cf. Module 06)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
for d in DIFFS:
    print(f"{d:7s}: " + "  ".join(f"budget={b}: {a:.2f}" for b, a in zip(BUDGETS, acc_c[d])))
print("定性对照：easy 曲线平（早停无损）、hard 曲线升（需要长思考）⇒ 06 模块结论成立")

## 6 · 实验 D（对应 07）：GSM-Symbolic 式扰动

**模拟预测**（07 模块）：模板记忆型模型在**换数字**后大幅掉点，**加无关从句**（GSM-NoOp）后掉得更狠；真正会算术的模型基本不掉 [Mirzadeh 2024]。

**设计**：题库每题生成两个变体——`number_swap`（用 `alt_params` 重新实例化模板，答案随之重算）与 `noop_clause`（追加一句带数字的无关信息，数字是故意放的：看模型会不会被勾走）。每版本每题采 4 条，对比三版准确率。**看什么**：掉点幅度与方向。前沿模型在小学题上通常只轻微掉点，且 NoOp 的影响常大于换数字——与 Mirzadeh 2024 定性一致；mock 路径按 ×0.85 / ×0.70 惩罚复现"脆弱模型"。

In [ ]:
# 实验 D：原版 vs 换数字 vs 加无关从句
VARIANTS = ["original", "number_swap", "noop_clause"]
acc_d = {}
for variant in VARIANTS:
    hits = []
    for prob in PROBLEMS:
        if variant == "original":
            q, ans = prob["question"], prob["answer"]
        elif variant == "number_swap":
            q, ans = prob["q_swap"], prob["a_swap"]
        else:
            q, ans = prob["q_noop"], prob["answer"]
        texts = llm_generate(solve_prompt(q), n=4, temperature=1.0)
        hits += [is_correct(extract_answer(t), ans) for t in texts]
    acc_d[variant] = float(np.mean(hits))

df_d = pd.Series(acc_d, name="accuracy").to_frame()
df_d["drop_vs_original"] = (df_d["accuracy"] - acc_d["original"]).round(3)
print(df_d.round(3))
print(f"\n示例 NoOp 变体：{PROBLEMS[0]['q_noop']}")
print("定性对照：换数字小掉、NoOp 掉更多 ⇒ 与 GSM-Symbolic 的发现一致（07 模块）")

## 7 · 汇总：模拟预测 vs 真模型观测 + 成本核算

四个实验各占一行，外加 05 模块的 **scaling 视角对照**——不用跑新实验：把实验 A 的 x 轴从"采样数 N"换算成"总 token 数"（N × 平均每条 output token），它就是一条真实的 inference scaling 曲线 [Brown 2024 的 coverage ~ log N 幂律]。一次采集、多次分析，是评测里的省钱习惯。

对照表只填三档结论：**一致 / 偏离 / 无法判定**。偏离时先怀疑抽取器与题库饱和，再怀疑模型；全对导致的平线属于"无法判定"——正确反应是换更难的题库（benchmark saturation 的桌面版）。最后从 `USAGE` 核算本次实验的真实 token 用量与等效成本——缓存目录就是完整账本。

In [ ]:
# 模拟预测 vs 真模型观测 + token 成本
avg_out_tokens = float(np.mean([len(t) // 4 for ts in samples.values() for t in ts]))
summary = pd.DataFrame([
    {"实验": "A · self-consistency", "模块": "02",
     "模拟预测": "majority@N 上升收敛；系统性偏差使难题 plateau",
     "真模型观测": f"@1={overall_a.loc[1]:.2f} → @16={overall_a.loc[16]:.2f}（hard@16={strat_a.loc[16, 'hard']:.2f}）"},
    {"实验": "B · verifier BoN", "模块": "03",
     "模拟预测": "verifier 优于随机时 BoN ≥ majority > random",
     "真模型观测": f"BoN={df_b['BoN(verifier)'].mean():.2f} / maj={df_b['majority@8'].mean():.2f} / rand={df_b['random(1)'].mean():.2f}"},
    {"实验": "(对照) inference scaling", "模块": "05",
     "模拟预测": "accuracy ~ log N 近线性后饱和（Brown 2024）",
     "真模型观测": f"即实验 A 曲线：x 轴换算为 N × {avg_out_tokens:.0f} token/条"},
    {"实验": "C · thinking budget", "模块": "06",
     "模拟预测": "简单题小预算即饱和；难题随预算上升",
     "真模型观测": f"easy {acc_c['easy'][0]:.2f}→{acc_c['easy'][-1]:.2f}；hard {acc_c['hard'][0]:.2f}→{acc_c['hard'][-1]:.2f}"},
    {"实验": "D · 扰动鲁棒性", "模块": "07",
     "模拟预测": "换数字小幅掉点；无关从句掉点更大",
     "真模型观测": f"orig {acc_d['original']:.2f} / swap {acc_d['number_swap']:.2f} / noop {acc_d['noop_clause']:.2f}"},
])
pd.set_option("display.max_colwidth", 58)
print("后端：", f"Gemini {MODEL}" if USE_REAL_API else "MockGemini（无 key 离线模拟）")
print(summary.to_string(index=False))

total_tokens = USAGE["prompt_tokens"] + USAGE["output_tokens"] + USAGE["thinking_tokens"]
cost = (USAGE["prompt_tokens"] / 1000 * PRICE_PER_1K["in"]
        + (USAGE["output_tokens"] + USAGE["thinking_tokens"]) / 1000 * PRICE_PER_1K["out"])
print(f"\nAPI 调用 {USAGE['calls']} 次（缓存命中 {USAGE['cache_hits']} 次）")
print(f"token 总计 {total_tokens:,}（prompt {USAGE['prompt_tokens']:,} / output {USAGE['output_tokens']:,}"
      f" / thinking {USAGE['thinking_tokens']:,}）")
print(f"等效成本 ≈ ${cost:.4f}" + ("" if USE_REAL_API else "（mock：token 为估算值，实际 $0）"))

---
## ✏️ 练习 1：实现 cache key 与缓存读写

不翻上文，自己实现 API 评测的缓存层三件套：

- `cache_key_ex(prompt, model, n, temperature, thinking_budget)` —— 把 5 个参数装进 dict，`json.dumps(..., sort_keys=True, ensure_ascii=False)` 序列化后取 `sha256` 的完整 hexdigest（64 字符）。**所有影响输出的参数都必须进 key**。
- `cache_write_ex(cache_dir, key, obj)` —— 把 `obj` 写到 `cache_dir/{key}.json`（`ensure_ascii=False`）。
- `cache_read_ex(cache_dir, key)` —— 读回 `obj`；文件不存在返回 `None`（不要抛异常）。

**提示**：`hashlib.sha256(blob.encode("utf-8")).hexdigest()`；路径拼接用 `os.path.join`；读取前先 `os.path.exists`。

In [ ]:
def cache_key_ex(prompt, model, n, temperature, thinking_budget):
    # TODO: dict -> json.dumps(sort_keys=True, ensure_ascii=False) -> sha256 hexdigest（64 字符）
    raise NotImplementedError

def cache_write_ex(cache_dir, key, obj):
    # TODO: 写入 cache_dir/{key}.json，ensure_ascii=False
    raise NotImplementedError

def cache_read_ex(cache_dir, key):
    # TODO: 读回 obj；文件不存在返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
import tempfile
k1 = cache_key_ex("解题", "gemini-2.5-flash", 4, 1.0, None)
assert isinstance(k1, str) and len(k1) == 64
assert k1 == cache_key_ex("解题", "gemini-2.5-flash", 4, 1.0, None)        # 确定性
assert cache_key_ex("解题", "gemini-2.5-flash", 4, 0.0, None) != k1        # temperature 必须进 key
assert cache_key_ex("解题", "gemini-2.5-flash", 4, 1.0, 512) != k1         # thinking_budget 必须进 key
assert cache_key_ex("解题", "gemini-2.5-flash", 8, 1.0, None) != k1        # n 必须进 key
with tempfile.TemporaryDirectory() as d:
    cache_write_ex(d, k1, {"texts": ["答案是 8。"]})
    assert cache_read_ex(d, k1) == {"texts": ["答案是 8。"]}
    assert cache_read_ex(d, "no_such_key") is None
print("✅ 练习 1 通过")

---
## ✏️ 练习 2：extract_answer 的 corner cases

实现 `extract_answer_ex(text)`，要求与正文同样的两级策略，并通过更刁钻的 corner cases：

1. 优先匹配 `答案是 / 答案为 / answer is`（大小写不敏感）后面的数值，**多次出现取最后一次**；
2. 千分位逗号要先剥掉（`"答案是 1,200"` → `1200.0`）；
3. 模式后面不是数字时（如 `"答案是 X"`），**退化**到全文最后一个数字；
4. 全文无数字返回 `None`；支持负数与小数。

**提示**：先 `text.replace(",", "")`；模式用 `re.findall(r"(?:答案是|答案为|answer is)\s*[:：]?\s*(-?\d+(?:\.\d+)?)", t, flags=re.IGNORECASE)` 的思路；退化路径 `re.findall(r"-?\d+(?:\.\d+)?", t)` 取最后一个。

In [ ]:
def extract_answer_ex(text):
    # TODO: 两级策略——先找'答案是/答案为/answer is X'（取最后一次），再退化到最后一个数字；失败返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert extract_answer_ex("经过计算，答案是 42。") == 42.0
assert extract_answer_ex("答案为 -3.5") == -3.5
assert extract_answer_ex("So the answer is 7.") == 7.0                     # 英文 + 大小写不敏感
assert extract_answer_ex("总收入 1,200 美元。答案是 1,200。") == 1200.0       # 千分位逗号
assert extract_answer_ex("先算出 5，修正后答案是 9。") == 9.0                 # 取最后一次出现的模式
assert extract_answer_ex("结果等于 17") == 17.0                             # 无模式 → 最后一个数字
assert extract_answer_ex("答案是 X（未能求出），中间步骤得到 12") == 12.0      # 模式后非数字 → 退化
assert extract_answer_ex("这题我不会。") is None
print("✅ 练习 2 通过")

---
## ✏️ 练习 3：实现 cost_estimate_ex 预算函数

实现 `cost_estimate_ex(prompts, n_samples, est_output_tokens, price_in_per_1k, price_out_per_1k)`，返回预估美元成本（float）：

- 每条 prompt 的输入 token 估算为 `max(1, len(p) // 4)`；
- 每条 prompt 要采 `n_samples` 次，每次输出按 `est_output_tokens` 估算；
- 成本 = 输入 token 总数 / 1000 × `price_in_per_1k` + 输出 token 总数 / 1000 × `price_out_per_1k`；
- 空列表返回 `0.0`。

**提示**：输入总 token = `sum(max(1, len(p) // 4) for p in prompts) * n_samples`；输出总 token = `est_output_tokens * len(prompts) * n_samples`。先算账再花钱——这是 API 评测的第一纪律。

In [ ]:
def cost_estimate_ex(prompts, n_samples, est_output_tokens, price_in_per_1k, price_out_per_1k):
    # TODO: 按 len(p)//4 估输入 token，按 est_output_tokens 估输出，折算成美元
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# "x"*400 -> 100 input tokens；"y"*100 -> 25 input tokens
c = cost_estimate_ex(["x" * 400, "y" * 100], 10, 200, 0.0003, 0.0025)
# in: (100+25)*10 = 1250 -> $0.000375；out: 200*2*10 = 4000 -> $0.01
assert abs(c - 0.010375) < 1e-9
assert cost_estimate_ex([], 10, 200, 0.0003, 0.0025) == 0.0
assert abs(cost_estimate_ex(["x" * 400], 20, 200, 0.0003, 0.0025)
           - 2 * cost_estimate_ex(["x" * 400], 10, 200, 0.0003, 0.0025)) < 1e-12   # n 翻倍成本翻倍
assert abs(cost_estimate_ex(["x" * 400], 1, 0, 0.0003, 0.0025) - 100 / 1000 * 0.0003) < 1e-12
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def cache_key_ex(prompt, model, n, temperature, thinking_budget):
    blob = json.dumps({"prompt": prompt, "model": model, "n": n,
                       "temperature": temperature, "thinking_budget": thinking_budget},
                      sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()

def cache_write_ex(cache_dir, key, obj):
    with open(os.path.join(cache_dir, key + ".json"), "w") as f:
        json.dump(obj, f, ensure_ascii=False)

def cache_read_ex(cache_dir, key):
    path = os.path.join(cache_dir, key + ".json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def extract_answer_ex(text):
    if not text:
        return None
    t = text.replace(",", "")
    m = re.findall(r"(?:答案是|答案为|answer is)\s*[:：]?\s*(-?\d+(?:\.\d+)?)", t, flags=re.IGNORECASE)
    if m:
        return float(m[-1])
    nums = re.findall(r"-?\d+(?:\.\d+)?", t)
    return float(nums[-1]) if nums else None

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def cost_estimate_ex(prompts, n_samples, est_output_tokens, price_in_per_1k, price_out_per_1k):
    if not prompts:
        return 0.0
    t_in  = sum(max(1, len(p) // 4) for p in prompts) * n_samples
    t_out = est_output_tokens * len(prompts) * n_samples
    return t_in / 1000 * price_in_per_1k + t_out / 1000 * price_out_per_1k

---
## 小结

- **同一套实验逻辑，后端可插拔**：`llm_generate` 把真 Gemini 与 MockGemini 收敛到一个接口——缓存、重试、成本核算都写在后端无关的层里。换本地模型（vLLM / Ollama 的 OpenAI 兼容端点）只需重写一个函数。
- **模拟 → 真实的对照纪律**：对照的是定性形状不是数值；20 题 × 1 模型只能下"形状复现 / 偏离 / 无法判定"三档结论，偏离先查抽取器与题库饱和，再怀疑模型。
- **API 评测四纪律**：成本预估先行、缓存 key 含全部采样参数、429/5xx 指数退避、key 永不落盘。缓存目录就是实验账本。
- **thinkingBudget 是 budget forcing 的产品化**：06 模块在模拟器里手搓的预算控制，在 gemini-2.5/3 上是一个 API 参数；报告时记实际 `thoughtsTokenCount` 而非名义预算。
- 想升级成研究项目：换 AIME / GSM-Symbolic 公开切片做题库、跨模型对照、verifier prompt 工程、token-per-correct-answer 的帕累托核算（见讲解第 8 节）。